# **Maestría en Inteligencia Artificial Aplicada**
## **Curso: Procesamiento de Lenguaje Natural (NLP)**
### Tecnológico de Monterrey
### Prof Luis Eduardo Falcón Morales

## **Actividad de la Semana: Análisis de Sentimiento**
### **Pre-procesamiento, Matrices Documento-Término (DTM) y TF-IDF.**

* **Nombre:** CarLuz
* **Matrícula:** A01796921

En esta actividad se utilizan datos de tres archivos del repositorio UCI:

* **amazon_cells_labelled.txt** — 1000 comentarios de Amazon.
* **imdb_labelled.txt** — 1000 comentarios de IMDB (748 registros visibles por un error de parseo).
* **yelp_labelled.txt** — 1000 comentarios de Yelp.

Fuente: https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences

### Importación de librerías

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTACIONES
# Cargamos todas las librerías al inicio para tener claras las dependencias.
# ─────────────────────────────────────────────────────────────────────────

import pandas as pd              # Manejo de DataFrames: carga y manipulación de archivos .txt
import numpy as np               # Operaciones numéricas y arreglos para matrices y gráficas
import re                        # Expresiones regulares: limpiar texto (quitar signos, espacios extra)
import string                    # Utilidades de cadenas (disponible para uso complementario)
import matplotlib.pyplot as plt  # Visualización: gráficas de frecuencias y nubes de palabras
from collections import Counter  # Estructura especializada en contar elementos (diccionario de frecuencias)

import nltk
nltk.download('punkt')           # Tokenizador de NLTK basado en modelo no supervisado
nltk.download('stopwords')       # Corpus de palabras vacías en múltiples idiomas
from nltk.corpus import stopwords      # Lista de stopwords de NLTK
from nltk.stem import SnowballStemmer  # Stemmer Snowball (Porter2): reduce palabras a su raíz morfológica

## **Pregunta - 1:**

* ### ¿Qué significa un Falso-Negativo y un Falso-Positivo en este problema? ¿Qué implicaciones podrían tener cada uno?
* ### ¿Cuál tipo de error se podría considerar más grave? Justifica tu respuesta.

##### **COMENTARIOS — Pregunta 1:**

En este problema los comentarios están etiquetados como **positivos (1)** o **negativos (0)**.

**Falso Positivo (FP):** El modelo predice **positivo** cuando el comentario es **negativo**.
> Ejemplo: clasificar *"Terrible product, broke after one day"* como reseña positiva.
> Implicación: comentarios negativos reales pasan desapercibidos; la empresa no detecta insatisfacción del cliente.

**Falso Negativo (FN):** El modelo predice **negativo** cuando el comentario es **positivo**.
> Ejemplo: clasificar *"Absolutely love this phone"* como negativa.
> Implicación: se subestima la satisfacción del cliente; posible pérdida de análisis de fortalezas del producto.

**¿Cuál es más grave?**

Con el dataset perfectamente balanceado (1500 positivos / 1500 negativos), ambos errores tienen el mismo peso estadístico. Sin embargo, en un contexto de **monitoreo de reputación de marca** (Amazon, Yelp, IMDB), el **Falso Positivo** es más costoso: comentarios negativos reales no se detectan, impidiendo una respuesta oportuna al cliente. Si el sistema filtra contenido, el Falso Negativo sería más costoso. En conclusión, la gravedad depende del objetivo de negocio del sistema.

### Stopwords — Construcción de `mystopwords`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VISUALIZACIÓN DE STOPWORDS ESTÁNDAR DE NLTK
# Mostramos la lista predeterminada para identificar qué palabras incluye.
# NLTK incluye negaciones ("not", "no", "don't") que necesitamos preservar
# para el análisis de sentimiento.
# ─────────────────────────────────────────────────────────────────────────

print(stopwords.words('english'))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONSTRUCCIÓN DE MYSTOPWORDS — Lista modificada que preserva negaciones
#
# PROBLEMA: NLTK incluye "not", "no", "don't" como stopwords porque en
# tareas generales no aportan significado. En análisis de sentimiento:
#   "not good" ≠ "good"
# Si eliminamos "not", perdemos la señal negativa del comentario.
#
# SOLUCIÓN: crear mystopwords excluyendo todas las formas negativas.
# ─────────────────────────────────────────────────────────────────────────

# Lista de palabras negativas que NO deben eliminarse del texto:
negwords = [
    'no', 'nor', 'not', 'ain', 'aren', "aren't",
    'don', "don't", 'couldn', "couldn't", 'didn', "didn't",
    'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't",
    'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't",
    'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't",
    'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't",
    'won', "won't", 'wouldn', "wouldn't"
]

# Comprensión de lista: recorre cada palabra w de la lista de NLTK
# y la conserva SOLO si NO está en negwords.
# Resultado: stopwords estándar de NLTK MENOS las negaciones.
mystopwords = [w for w in stopwords.words('english') if w not in negwords]

print("Total de stopwords en la lista original de NLTK: %d" % len(stopwords.words('english')))
print("Total de stopwords excluyendo los conectivos negativos: %d\n" % len(mystopwords))
print(mystopwords)

### Carga de datos

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CARGA DE LOS TRES ARCHIVOS DE DATOS
#
# Formato de cada archivo: dos columnas separadas por tabulador (\t)
#   Columna 1: comentario (review)
#   Columna 2: etiqueta (label) — 0 = negativo, 1 = positivo
#
# Parámetros de pd.read_csv:
#   sep / delimiter = '\t'  → separador tabulador
#   names = [...]           → nombres de columnas (el archivo no tiene encabezado)
#   header = None           → confirma que la primera fila es dato, no encabezado
#   encoding = 'utf-8'      → codificación de caracteres
# ─────────────────────────────────────────────────────────────────────────

# Ajustar la ruta según el entorno (Google Colab, local, etc.):
dfa = pd.read_csv('amazon_cells_labelled.txt', sep='\t', names=['review','label'], header=None, encoding='utf-8')
dfi = pd.read_csv('imdb_labelled.txt', delimiter='\t', names=['review','label'], header=None, encoding='utf-8')
dfy = pd.read_csv('yelp_labelled.txt', sep='\t', names=['review','label'], header=None, encoding='utf-8')

print('Total de registros de Amazon:', dfa.shape)   # esperado: (1000, 2)
print('Total de registros de IMBD:', dfi.shape)     # aparece: (748, 2) — problema a corregir
print('Total de registros de Yelp:', dfy.shape)     # esperado: (1000, 2)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VERIFICACIÓN INICIAL — primeras filas de Amazon
# dfa.head() muestra las primeras 5 filas del DataFrame.
# Confirma que tiene las columnas review (texto) y label (0 o 1).
# ─────────────────────────────────────────────────────────────────────────

dfa.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# DETECCIÓN DEL PROBLEMA EN IMDB
# .values.tolist() convierte el DataFrame a lista Python para inspeccionarlo.
# [17:21] muestra los registros de índice 17 al 20.
# El registro 19 contiene decenas de comentarios pegados en una sola celda:
# ese es el problema que corregiremos en la Pregunta 2.
# ─────────────────────────────────────────────────────────────────────────

dfi.values.tolist()[17:21]

## **Pregunta - 2:**

Corrección del DataFrame de IMDB para obtener los 1000 comentarios correctamente separados.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CORRECCIÓN DEL DATAFRAME DE IMDB: 748 → 1000 registros
#
# PROBLEMA: pd.read_csv detecta solo 748 filas. Algunos comentarios de IMDB
# contienen caracteres especiales (comillas tipográficas, guiones Unicode)
# que confunden al parser de pandas: interpreta múltiples comentarios
# como una sola celda gigante. El registro 19 llega a contener
# más de 60 comentarios pegados en una sola fila.
#
# FORMATO CORRECTO por línea:
#   comentario\tetiqueta\n
# El separador entre comentario y etiqueta es SIEMPRE el último tabulador.
#
# ESTRATEGIA: leer el archivo como texto plano completo (f.read())
# y parsear manualmente cada línea. Esto da control total sobre la separación.
# ─────────────────────────────────────────────────────────────────────────

newdfi = []  # lista vacía que recibirá los 1000 pares [comentario, etiqueta]

# PASO 1: Leer el archivo completo como un solo string de texto.
# errors='replace' → si hay bytes no-UTF8, los reemplaza con el símbolo de
# reemplazo Unicode en lugar de lanzar un error de codificación.
with open('imdb_labelled.txt', 'r', encoding='utf-8', errors='replace') as f:
    content = f.read()  # el archivo completo como un único string

# PASO 2: Separar por salto de línea real (\n).
# Esto "desdobla" los registros pegados: cada comentario queda como una línea separada.
for line in content.split('\n'):

    line = line.strip()  # quita espacios, tabulaciones y \r (retorno de carro de Windows)

    if line and '\t' in line:  # procesar solo líneas no vacías que tengan tabulador

        # rsplit('\t', 1): divide por el ÚLTIMO tabulador (maxsplit=1).
        # Se usa rsplit en lugar de split para que si el comentario tiene
        # tabuladores internos, solo se separe en el tabulador de la etiqueta.
        parts = line.rsplit('\t', 1)

        if len(parts) == 2:               # verificar que hay exactamente 2 partes
            review = parts[0].strip()     # parte izquierda = texto del comentario
            try:
                label = int(parts[1].strip())  # parte derecha = etiqueta 0 o 1
                if review:                     # solo si el comentario no está vacío
                    newdfi.append([review, label])  # agregar el par a la lista
            except ValueError:
                pass  # ignorar líneas cuya etiqueta no sea un entero válido

print('Total de comentarios de la lista errónea original de IMBD:', (len(dfi)))
print('Total de comentarios de la nueva lista corregida de IMBD:', (len(newdfi)))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VERIFICACIÓN DE LA CORRECCIÓN
# Los registros 17-20 deben mostrar ahora comentarios individuales.
# Antes: el índice 19 contenía 60+ comentarios pegados.
# Después: cada posición debe tener un solo comentario con su etiqueta.
# ─────────────────────────────────────────────────────────────────────────

newdfi[17:21]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONVERTIR LA LISTA CORREGIDA A DATAFRAME
# pd.DataFrame(lista, columns=[...]) crea el DataFrame con nombres de columna.
# dfii.info() muestra tipos de datos, nulos y dimensiones del nuevo DataFrame.
# dfii.head() muestra las primeras 5 filas para validación visual.
# ─────────────────────────────────────────────────────────────────────────

dfii = pd.DataFrame(newdfi, columns=['review','label'])

dfii.info()
dfii.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONCATENAR LOS TRES DataFrames EN UNO SOLO (3000 registros)
#
# pd.concat([lista_de_dfs], ignore_index=True):
#   - apila los tres DataFrames verticalmente (por filas)
#   - ignore_index=True: reinicia el índice de 0 a 2999 en lugar de
#     mantener tres rangos 0-999 repetidos que causarían índices duplicados
#
# df.info() debe reportar: 3000 filas, 2 columnas, 0 nulos.
# ─────────────────────────────────────────────────────────────────────────

df = pd.concat([dfa, dfii, dfy], ignore_index=True)  # dataset completo de 3000 comentarios
df.info()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# DISTRIBUCIÓN DE CLASES
# value_counts() cuenta cuántos registros hay de cada etiqueta.
# Esperamos 1500 positivos (1) y 1500 negativos (0):
# dataset perfectamente balanceado — no hay sesgo de clase.
# ─────────────────────────────────────────────────────────────────────────

df['label'].value_counts()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VISTA INICIAL DEL DATASET COMBINADO
# df.head() muestra las primeras 5 filas del dataset final de 3000 registros.
# ─────────────────────────────────────────────────────────────────────────

df.head()

## **Pregunta - 3:**

Tratamiento de los registros 1125 y 1788 (`"10/10"`).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# INSPECCIÓN DE LOS REGISTROS PROBLEMÁTICOS
#
# df.iloc[fila, :]:
#   .iloc  → acceso posicional por índice entero (no por nombre de columna)
#   fila   → número de fila (1125 o 1788)
#   :      → seleccionar TODAS las columnas del registro
#
# Resultado esperado: ambos tienen review='10/10' y label=1 (positivo).
# Después de la limpieza (que elimina caracteres no alfabéticos),
# "10/10" quedará completamente vacío → no hay tokens para analizar.
# ─────────────────────────────────────────────────────────────────────────

print(df.iloc[1125,:])
print(df.iloc[1788,:])

##### **COMENTARIOS — Pregunta 3:**

Los comentarios `"10/10"` quedarán **completamente vacíos** después de la limpieza (la función `clean_tok` solo conserva caracteres alfabéticos, y "10/10" no tiene ninguno).

**Decisión: descartarlos implícitamente.**

**Justificación:**
- Un comentario vacío produce un **vector de ceros** en la DTM — no aporta ninguna señal al modelo.
- Aunque tienen etiqueta positiva (1), sin tokens el modelo no puede aprender nada de ellos.
- Son solo **2 de 3000 registros (0.067%)** — su eliminación no afecta el balance del dataset ni la capacidad de generalización del modelo.
- La alternativa de reemplazarlos por "excellent" o "perfect" introduciría **información artificial** no original del usuario, lo cual es metodológicamente menos riguroso.

En la práctica, estos registros quedan como listas vacías `[]` en `Xcleantok`. Al hacer el `join` posterior generan strings vacíos `""` que `CountVectorizer` maneja sin errores (produce vectores de ceros). No es necesario eliminarlos explícitamente.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# SEPARAR ENTRADA (X) Y SALIDA (y)
#
# X: Serie de 3000 strings — los comentarios de texto (entrada del modelo)
# y: Serie de 3000 enteros 0/1 — etiquetas de sentimiento (salida a predecir)
#
# assert: instrucción de verificación que lanza AssertionError si la condición
# no se cumple. Usamos assert para confirmar que tenemos exactamente 3000
# registros en cada Serie antes de continuar con el pipeline.
# ─────────────────────────────────────────────────────────────────────────

X = df.review   # Serie de strings: los comentarios de texto
y = df.label    # Serie de enteros: 0 (negativo) o 1 (positivo)

assert X.shape == (3000,)  # verificar dimensión de X — debe ser (3000,)
assert y.shape == (3000,)  # verificar dimensión de y — debe ser (3000,)

## **Pregunta - 4:** Limpieza y tokenización — `clean_tok()`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FUNCIÓN clean_tok() — LIMPIEZA Y TOKENIZACIÓN BÁSICA
#
# Recibe: doc → string con el comentario original
# Devuelve: lista de tokens limpios
#
# ORDEN DE LOS PASOS (justificado):
# 1. Eliminar caracteres no alfabéticos PRIMERO con espacio como reemplazo
#    → evita que "minutes.MAJOR" quede como "minutesMAJOR" (token único erróneo)
# 2. Colapsar espacios múltiples generados en el paso anterior
# 3. Minúsculas ANTES de comparar con stopwords (que están en minúsculas)
# 4. Tokenización: split() divide por espacios
# 5. Filtrar stopwords (preservando negaciones en mystopwords)
# 6. Filtro de longitud: descartar tokens de 1 sola letra
# ─────────────────────────────────────────────────────────────────────────

def clean_tok(doc):

    # PASO 1 — Solo caracteres alfabéticos.
    # re.sub(patrón, reemplazo, texto): busca todas las ocurrencias del patrón
    # y las reemplaza por el string indicado.
    # r'[^a-zA-Z]' → raw string (la 'r' evita que Python interprete '\' como escape).
    # [^...] = NEGACIÓN del conjunto. [^a-zA-Z] = cualquier caracter que NO sea
    # letra A-Z mayúscula o minúscula: números, puntos, comas, !, ?, guiones, etc.
    # Se reemplaza por ' ' (espacio) y NO por '' (vacío), para que palabras
    # separadas solo por signos no queden pegadas:
    #   "minutes.MAJOR" → "minutes MAJOR" (correcto con espacio)
    #   "minutes.MAJOR" → "minutesMAJOR"  (incorrecto con vacío)
    doc = re.sub(r'[^a-zA-Z]', ' ', doc)

    # PASO 2 — Eliminar espacios múltiples.
    # doc.strip(): elimina espacios al inicio y al final del string.
    # r'\s{2,}': \s = cualquier espacio en blanco; {2,} = 2 o más repeticiones.
    # Colapsa todos los espacios extras generados en el paso 1 en un solo espacio.
    doc = re.sub(r'\s{2,}', ' ', doc.strip())

    # PASOS 3 y 4 — Minúsculas + tokenización en una sola línea.
    # doc.lower(): convierte a minúsculas → "Good" == "good" == "GOOD"
    # .split(): divide el string por espacios y devuelve lista de tokens.
    #   Sin argumento, split() elimina automáticamente los elementos vacíos.
    tokens = doc.lower().split()

    # PASO 5 — Eliminar stopwords.
    # Comprensión de lista: conserva w solo si NO está en mystopwords.
    # mystopwords preserva negaciones (not, no, don't...) clave para sentimiento.
    # Se eliminan: "the", "a", "is", "was", "for", "to", etc.
    tokens = [w for w in tokens if w not in mystopwords]

    # PASO 6 — Filtro de longitud mínima.
    # len(w) > 1: descarta tokens de 1 solo caracter (letras sueltas como
    # "a", "i", "s", "t") que pueden sobrevivir el filtro de stopwords
    # o generarse como residuo del proceso de limpieza regex.
    tokens = [w for w in tokens if len(w) > 1]

    return tokens  # lista final de tokens limpios

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# APLICAR clean_tok() A TODOS LOS COMENTARIOS
#
# Comprensión de lista: aplica clean_tok(x) a cada comentario x de la Serie X.
# Resultado: Xcleantok es una lista de 3000 listas de tokens.
#   Xcleantok[0] = lista de tokens del primer comentario
#   Xcleantok[i] = lista de tokens del comentario i
# ─────────────────────────────────────────────────────────────────────────

Xcleantok = [clean_tok(x) for x in X]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VERIFICACIÓN: primeros 5 comentarios tokenizados.
# Confirmar que:
#   ✓ No hay signos de puntuación, números ni mayúsculas
#   ✓ No hay stopwords generales (the, a, is, was, for...)
#   ✓ Sí se conservan las negaciones (no, not, don't...)
#   ✓ El 4to comentario: "minutes" y "major" están separados (no "minutesmajor")
#   ✓ Los registros 1125 y 1788 ("10/10") aparecen como listas vacías []
# ─────────────────────────────────────────────────────────────────────────

for x in Xcleantok[0:5]:
    print(x)
print()
print('Registro 1125 (10/10):', Xcleantok[1125])
print('Registro 1788 (10/10):', Xcleantok[1788])

## **Pregunta - 5:** Limpieza adicional — `clean_doc()`

Se aplican dos procesos de normalización adicionales:

1. **Stemming con SnowballStemmer (inglés):** Reduce cada token a su raíz morfológica.
   Ej: `"loving"` → `"love"`, `"movies"` → `"movi"`, `"conversations"` → `"convers"`.
   Agrupa variantes de la misma palabra en un solo token, reduciendo el vocabulario.

2. **Filtro de longitud post-stemming:** El stemming puede producir raíces de 1 sola letra
   sin valor semántico (ej: `"be"` → `"b"`). Se filtran para mantener solo tokens con ≥ 2 caracteres.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FUNCIÓN clean_doc() — NORMALIZACIÓN ADICIONAL
#
# Recibe: doc → lista de tokens ya limpios de clean_tok()
# Devuelve: lista de tokens normalizados (raíces morfológicas)
#
# PROCESO ADICIONAL 1 — Stemming con SnowballStemmer:
#   Elimina sufijos morfológicos para reducir variantes al mismo concepto.
#   Ejemplos:
#     "excellent"     → "excel"
#     "conversations" → "convers"
#     "lasting"       → "last"
#     "great"         → "great"  (ya es raíz, no cambia)
#
# PROCESO ADICIONAL 2 — Filtro de longitud > 1 post-stemming:
#   El stemming puede reducir palabras de 2 letras a raíces de 1 letra.
#   Ej: "be" → "b". Estas raíces de 1 caracter se descartan por no tener
#   valor semántico en el contexto de sentimiento.
# ─────────────────────────────────────────────────────────────────────────

# Instanciar el stemmer UNA SOLA VEZ, fuera de la función.
# SnowballStemmer('english'): algoritmo Porter2, más preciso que el Porter original.
# Se crea fuera del bucle para no recrear el objeto en cada una de las 3000 llamadas.
snow_stemmer = SnowballStemmer('english')

def clean_doc(doc):

    # PROCESO ADICIONAL 1: Stemming.
    # [snow_stemmer.stem(w) for w in doc]: aplica stem() a cada token w.
    # stem(w) devuelve la raíz morfológica del token usando el algoritmo Snowball.
    tokens = [snow_stemmer.stem(w) for w in doc]

    # PROCESO ADICIONAL 2: Filtro de longitud mínima post-stemming.
    # Descarta raíces de 1 sola letra que el stemming puede generar.
    # Ejemplo: "be" → "b" → descartado (len("b") = 1, no > 1)
    tokens = [w for w in tokens if len(w) > 1]

    return tokens

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# APLICAR clean_doc() A TODOS LOS COMENTARIOS TOKENIZADOS
#
# Comprensión de lista: aplica clean_doc(x) a cada lista de tokens x de Xcleantok.
# Resultado: Xclean es la representación final normalizada de los 3000 comentarios.
#   Xclean[i] = lista de raíces morfológicas del comentario i
# ─────────────────────────────────────────────────────────────────────────

Xclean = [clean_doc(x) for x in Xcleantok]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VERIFICACIÓN: primeros 5 comentarios después de clean_doc().
# Comparar con Xcleantok[0:5] para ver el efecto del stemming.
#   Xcleantok[1]: ['good', 'case', 'excellent', 'value']
#   Xclean[1]:    ['good', 'case', 'excel',     'valu' ]
# ─────────────────────────────────────────────────────────────────────────

print('--- Después de clean_tok (Xcleantok) ---')
for x in Xcleantok[0:5]:
    print(x)

print()
print('--- Después de clean_doc (Xclean) ---')
for x in Xclean[0:5]:
    print(x)

## **Pregunta - 6:** Nube de palabras por clase

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PARTE 1: CONSTRUIR STRINGS DE TOKENS POSITIVOS Y NEGATIVOS
#
# WordCloud necesita un string de texto, no una lista de listas.
# Estrategia:
#   1. Iterar simultáneamente los tokens (Xclean) y las etiquetas (y)
#   2. Clasificar cada token en su lista según la etiqueta del comentario
#   3. Unir con ' '.join() para crear el string que WordCloud procesa
# ─────────────────────────────────────────────────────────────────────────

pos_tok = []  # acumulador de tokens de todos los comentarios positivos
neg_tok = []  # acumulador de tokens de todos los comentarios negativos

# zip(Xclean, y): combina en pares (tokens_del_comentario_i, etiqueta_i)
# para iterar ambas listas simultáneamente con un solo for.
for x, c in zip(Xclean, y):
    if c == 1:
        # .extend(x): agrega CADA ELEMENTO de la lista x a pos_tok.
        # Diferente a .append(x), que agregaría la lista completa como un solo elemento.
        pos_tok.extend(x)
    else:
        neg_tok.extend(x)

# ' '.join(lista): une todos los tokens con un espacio entre ellos.
# WordCloud calcula las frecuencias a partir de este string.
pt = ' '.join(pos_tok)  # string con todos los tokens positivos
nt = ' '.join(neg_tok)  # string con todos los tokens negativos

print('Muestra de tokens positivos:', pt[:200])
print()
print('Muestra de tokens negativos:', nt[:200])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# PARTE 2: GENERAR LAS NUBES DE PALABRAS
#
# WordCloud: visualización donde el tamaño de cada palabra es proporcional
# a su frecuencia en el string de entrada.
# matplotlib: permite mostrar las dos nubes lado a lado en la misma figura.
# ─────────────────────────────────────────────────────────────────────────

from wordcloud import WordCloud
import matplotlib.pyplot as plt

# plt.subplots(1, 2, figsize=(14, 6)):
#   - 1 fila, 2 columnas de subgráficas
#   - figsize: ancho=14 pulgadas, alto=6 pulgadas
#   - ax1 → subplot izquierdo, ax2 → subplot derecho
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ── Nube de comentarios POSITIVOS ──────────────────────────────────────
# WordCloud(...): configura el generador de nubes.
#   width, height: dimensiones del canvas en píxeles
#   background_color='white': fondo blanco para mejor legibilidad
#   colormap='Greens': paleta de verdes → identifica visualmente la clase positiva
#   max_words=100: muestra las 100 palabras más frecuentes
# .generate(pt): calcula frecuencias y genera el objeto de nube desde el string pt
wc_pos = WordCloud(width=600, height=400, background_color='white',
                   colormap='Greens', max_words=100).generate(pt)

# ax1.imshow(): renderiza la imagen de la nube en el subplot izquierdo
#   interpolation='bilinear': suaviza los bordes de las letras al renderizar
ax1.imshow(wc_pos, interpolation='bilinear')
ax1.axis('off')                                      # oculta los ejes x/y (no tienen sentido aquí)
ax1.set_title('Comentarios Positivos', fontsize=14, fontweight='bold')

# ── Nube de comentarios NEGATIVOS ──────────────────────────────────────
# colormap='Reds': paleta de rojos → identifica visualmente la clase negativa
wc_neg = WordCloud(width=600, height=400, background_color='white',
                   colormap='Reds', max_words=100).generate(nt)
ax2.imshow(wc_neg, interpolation='bilinear')
ax2.axis('off')
ax2.set_title('Comentarios Negativos', fontsize=14, fontweight='bold')

# plt.tight_layout(): ajusta automáticamente los márgenes para que las
# dos nubes no se superpongan entre sí.
plt.tight_layout()
plt.show()

##### **COMENTARIOS — Pregunta 6 — parte 3:**

**Comentarios Positivos:** Palabras dominantes: `great`, `good`, `love`, `excel` (excellent), `best`, `work`. Son señales claras de satisfacción. También aparecen `film`, `phone`, `food` — palabras de contexto de dominio (IMDB, Amazon, Yelp) que aparecen en ambas clases.

**Comentarios Negativos:** La palabra dominante es `not` — confirma que preservar las negaciones en `mystopwords` fue la decisión correcta y crítica. Otras señales negativas: `bad`, `don`, `no`, `worst`, `disappoint`.

**Conclusión:** Palabras de contexto (`movi`, `phone`, `food`, `place`) aparecen en ambas nubes porque son neutras al sentimiento. El modelo las discrimina por co-ocurrencia con palabras de sentimiento, no de forma aislada.

### Train — Validation — Test Split

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# DIVISIÓN DE DATOS: TRAIN (70%) / VALIDATION (15%) / TEST (15%)
#
# Se realizan DOS llamadas a train_test_split:
#   1ra llamada: separa TRAIN (70%) del 30% restante
#   2da llamada: divide ese 30% en mitades iguales → VAL (15%) y TEST (15%)
#
# Parámetros:
#   train_size=0.70  → 70% de los datos para entrenamiento (2100 comentarios)
#   shuffle=True     → mezcla aleatoriamente antes de dividir, para que las
#                      tres fuentes (Amazon/IMDB/Yelp) queden distribuidas
#                      proporcionalmente en los tres conjuntos
#   random_state=1   → semilla fija para reproducibilidad
#                      (siempre produce la misma partición exacta)
# ─────────────────────────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split

# Primera división: 70% train, 30% restante
x_train, x_val_and_test, y_train, y_val_and_test = train_test_split(
    Xclean, y, train_size=.70, shuffle=True, random_state=1)

# Segunda división: el 30% restante se divide al 50% → 15% val, 15% test
x_val, x_test, y_val, y_test = train_test_split(
    x_val_and_test, y_val_and_test, test_size=.50, shuffle=True, random_state=1)

# los "x_" son listas de listas de tokens; los "y_" son Series de pandas
print('X,y Train:', len(x_train), len(y_train))    # esperado: 2100
print('X,y Val:', len(x_val), len(y_val))           # esperado: 450
print('X,y Test', len(x_test), len(y_test))         # esperado: 450

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONSTRUCCIÓN DEL DICCIONARIO DE FRECUENCIAS
#
# IMPORTANTE: se construye SOLO con x_train.
# Usar val o test contaminaría el modelo (data leakage): el modelo "vería"
# información de datos futuros durante el aprendizaje del vocabulario.
#
# Counter(): subclase de dict especializada en contar elementos.
#   Se inicializa vacío y se actualiza iterativamente.
#
# midiccionario.update(lista): suma la frecuencia de cada elemento de la lista.
#   Al terminar el bucle, midiccionario contiene la frecuencia acumulada
#   de cada token en TODO el corpus de entrenamiento.
# ─────────────────────────────────────────────────────────────────────────

from collections import Counter

midiccionario = Counter()  # inicializar contador vacío

# Recorrer cada uno de los 2100 comentarios de entrenamiento:
for k in range(len(x_train)):
    midiccionario.update(x_train[k])  # sumar frecuencias del comentario k

print('Longitud del diccionario:', len(midiccionario))  # tokens únicos en x_train
print('\n(word, frequency):')
print(midiccionario.most_common(10))  # los 10 tokens más frecuentes

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# GRÁFICA DE DISTRIBUCIÓN DE FRECUENCIAS DEL VOCABULARIO
#
# Visualiza la distribución de frecuencias del vocabulario.
# Se observa la "Ley de Zipf": pocos tokens son muy frecuentes,
# la mayoría aparece muy pocas veces (cola larga a la derecha).
# Esta observación motiva el filtro de frecuencia mínima (Pregunta 7).
# ─────────────────────────────────────────────────────────────────────────

plt.plot(list(np.arange(len(midiccionario))), list(midiccionario.values()), color='blue')
plt.title('Distribución de frecuencias del vocabulario')
plt.xlabel('Índice de token (ordenado por frecuencia)')
plt.ylabel('Frecuencia de aparición en train')
plt.show()

## **Pregunta - 7:** Frecuencia mínima del vocabulario

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FILTRO DE FRECUENCIA MÍNIMA — min_freq y midicc
#
# Muchos tokens aparecen solo 1 vez en el corpus de entrenamiento (hápax
# legomena). Estos tokens son difícilmente generalizables: pueden ser errores
# tipográficos, nombres propios únicos, o palabras que el modelo no verá
# de nuevo en val/test. Incluirlos infla el vocabulario sin aportar poder
# predictivo.
#
# Se elige min_freq = 2 porque:
#   - Elimina hápax legomena (frecuencia = 1) sin perder información valiosa
#   - Con min_freq=3: vocabulario cae a ~900 tokens y aparecen 7+ documentos
#     vacíos en train (indeseable para el entrenamiento)
#   - Con min_freq=2: 1450 tokens, balance óptimo tamaño vs calidad
# ─────────────────────────────────────────────────────────────────────────

# Frecuencia mínima de aparición para incluir un token en el vocabulario:
min_freq = 2

# Comprensión de diccionario: {clave: valor for k,v in items if condición}
# Conserva solo los tokens cuya frecuencia c es mayor o igual a min_freq.
midicc = {w: c for w, c in midiccionario.items() if c >= min_freq}

print('Nueva longitud del nuevo vocabulario:', len(midicc))
print(list(midicc.items())[0:5])  # muestra 5 pares (token, frecuencia) del vocabulario filtrado

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# FILTRAR LOS TRES CONJUNTOS CON EL VOCABULARIO REDUCIDO
#
# Para cada comentario s en cada conjunto, conservar solo los tokens
# que están en midicc (vocabulario con min_freq >= 2).
# Los tokens con frecuencia = 1 se descartan de todos los conjuntos.
#
# Comprensión de lista anidada (doble comprensión):
#   Lista externa: itera cada comentario s del conjunto
#   Lista interna: filtra los tokens de s según si están en midicc
# ─────────────────────────────────────────────────────────────────────────

# Filtrar cada conjunto por separado:
train_x = [[w for w in ss if w in midicc] for ss in x_train]
val_x   = [[w for w in ss if w in midicc] for ss in x_val]
test_x  = [[w for w in ss if w in midicc] for ss in x_test]

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# VERIFICACIÓN: comparar comentarios antes y después del filtro de min_freq.
# Observar qué tokens fueron eliminados (frecuencia < 2 en train).
# ─────────────────────────────────────────────────────────────────────────

for k in range(3):
    print('Antes:', x_train[k])    # tokens de clean_doc (sin filtro de frecuencia)
    print('Después:', train_x[k])  # tokens restantes según midicc
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONVERTIR LISTAS DE TOKENS A STRINGS PARA CountVectorizer
#
# sklearn.CountVectorizer recibe STRINGS de texto, no listas de tokens.
# ' '.join(lista): une todos los tokens de una lista con un espacio entre ellos.
#   Ejemplo: ['good', 'case', 'excel'] → 'good case excel'
#
# Esto convierte cada comentario de su representación como lista de tokens
# a un string listo para ser vectorizado por CountVectorizer.
# ─────────────────────────────────────────────────────────────────────────

train_x_docs = [' '.join(x) for x in train_x]  # 2100 strings de train
val_x_docs   = [' '.join(x) for x in val_x]    # 450 strings de validación
test_x_docs  = [' '.join(x) for x in test_x]   # 450 strings de prueba

# Verificar que los primeros comentarios son strings correctamente formados:
for k in range(3):
    print(train_x_docs[k])

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONSTRUCCIÓN DE LA DOCUMENT-TERM MATRIX (DTM) — MATRICES SPARSE COUNT
#
# CountVectorizer transforma strings en vectores numéricos.
# Cada posición del vector corresponde a un token del vocabulario.
# El valor es la frecuencia (conteo) de ese token en el documento.
#
# vocabulary=mivocab:
#   Fija las 1450 columnas de la matriz explícitamente.
#   Sin esto, CountVectorizer construiría su propio vocabulario desde los datos,
#   y las matrices de val/test podrían tener diferente dimensión.
#
# fit_transform(train_x_docs):
#   APRENDE el vocabulario Y transforma train en una sola llamada.
#
# transform(val/test):
#   SOLO transforma. No reajusta el vocabulario.
#   Esto es crucial para evitar data leakage: val y test deben tratarse
#   como "documentos nuevos desconocidos" para el modelo.
#
# Las matrices resultantes son SPARSE (scipy.sparse.csr_matrix):
#   almacenan solo los valores no-cero para ahorrar memoria RAM,
#   dado que ~99% de la DTM son ceros (la mayoría de tokens no aparece en cada doc).
# ─────────────────────────────────────────────────────────────────────────

from sklearn.feature_extraction.text import CountVectorizer

mivocab = list(midicc.keys())  # vocabulario como lista: define el orden de las columnas

# Crear el vectorizador con el vocabulario fijo de 1450 tokens:
countvectorizer = CountVectorizer(vocabulary=mivocab)

# Generar las tres matrices DTM sparse:
train_x_count = countvectorizer.fit_transform(train_x_docs)  # shape: (2100, 1450)
val_x_count   = countvectorizer.transform(val_x_docs)         # shape: (450, 1450)
test_x_count  = countvectorizer.transform(test_x_docs)        # shape: (450, 1450)

# Visualizar valores de la DTM (convertir sparse → array solo para visualización):
count_tokens = countvectorizer.get_feature_names_out()
df_countvect = pd.DataFrame(data=train_x_count.toarray(), columns=count_tokens)
print(df_countvect.iloc[0:3, 6:18])  # primeros 3 documentos, columnas 6 a 17

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# ANÁLISIS DE SPARSITY (DISPERSIÓN) DE LA DTM DE ENTRENAMIENTO
#
# Sparsity: porcentaje de valores cero en la matriz.
# En DTMs de texto es típico tener >99% de ceros porque ningún documento
# usa todas las palabras del vocabulario simultáneamente.
# Esto justifica usar matrices dispersas (sparse) que solo almacenan no-ceros.
#
# train_x_count.shape[0]  → filas (documentos): 2100
# train_x_count.shape[1]  → columnas (tokens): 1450
# N                        → total de celdas de la matriz
# count_nonzero()          → celdas con valor diferente de cero
# ─────────────────────────────────────────────────────────────────────────

# Total de entradas de la matriz (filas × columnas):
N = (train_x_count.shape[0] * train_x_count.shape[1])
print("Total de entradas de la matriz DTM de entrenamiento: %d" % N)

N_no_cero = train_x_count.count_nonzero()
print("Total de valores no cero: %d" % N_no_cero)
print("Total de valores cero: %d" % (N - N_no_cero))

# Porcentaje de sparsity:
p_sparse = 1 - train_x_count.count_nonzero() / (train_x_count.shape[0] * train_x_count.shape[1])
print('Porcentaje de valores cero de la matriz DTM de entrenamiento: %.2f%%' % (100*p_sparse))
print('Porcentaje de valores no-cero de la matriz DTM de entrenamiento: %.2f%%' % (100 - 100*p_sparse))

## **Pregunta - 8:** Matrices TF-IDF

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CONSTRUCCIÓN DE LAS MATRICES TF-IDF PARA TRAIN, VALIDACIÓN Y TEST
#
# TF-IDF (Term Frequency — Inverse Document Frequency):
# Representación alternativa a las frecuencias brutas que pondera cada token
# por qué tan RARO es en el corpus completo.
#
# Fórmulas:
#   TF(t, d)  = frecuencia del token t en el doc d / total de tokens en d
#   IDF(t)    = log( N / número de documentos que contienen t )
#   TF-IDF    = TF × IDF
#
# Intuición:
#   - Token frecuente en UN doc pero raro en el corpus → peso alto (informativo)
#   - Token frecuente en TODOS los docs → IDF bajo → peso bajo (poco discriminativo)
#
# Ejemplo con este dataset:
#   "movi"    aparece en ~500 de 2100 docs → IDF bajo → peso TF-IDF bajo
#   "jawbone" aparece en 2 de 2100 docs    → IDF alto → peso TF-IDF alto si aparece
#
# IMPORTANTE — Para evitar data leakage:
#   fit_transform sobre train: aprende los pesos IDF del corpus de entrenamiento
#   transform sobre val/test:  aplica los MISMOS pesos IDF sin recalcularlos
# ─────────────────────────────────────────────────────────────────────────

from sklearn.feature_extraction.text import TfidfVectorizer

# Crear el vectorizador TF-IDF con el mismo vocabulario de 1450 tokens:
tfidfvectorizer = TfidfVectorizer(vocabulary=mivocab)

# fit_transform sobre train: calcula pesos IDF Y transforma en vectores TF-IDF normalizados:
train_x_tfidf = tfidfvectorizer.fit_transform(train_x_docs)  # shape: (2100, 1450)

# transform sobre val y test: aplica los MISMOS pesos IDF aprendidos de train.
# No recalcula IDF con datos de val/test (eso sería data leakage).
val_x_tfidf   = tfidfvectorizer.transform(val_x_docs)         # shape: (450, 1450)
test_x_tfidf  = tfidfvectorizer.transform(test_x_docs)        # shape: (450, 1450)

# Visualizar algunos valores TF-IDF de los primeros 3 documentos:
# Los valores son decimales entre 0 y 1 (normalizados), a diferencia
# de las frecuencias enteras de Count.
tfidf_tokens = tfidfvectorizer.get_feature_names_out()
df_tfidfvect = pd.DataFrame(data=train_x_tfidf.toarray(), columns=count_tokens)
print(df_tfidfvect.iloc[0:3, 6:18])  # valores TF-IDF de los primeros 3 docs

## **Pregunta - 9:** Modelos con matrices Count

Se aplican Regresión Logística, Random Forest y Naive Bayes con las matrices de conteo.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# IMPORTAR CLASIFICADORES Y MÉTRICA DE EVALUACIÓN
# ─────────────────────────────────────────────────────────────────────────

from sklearn.linear_model import LogisticRegression     # clasificador lineal
from sklearn.ensemble import RandomForestClassifier     # ensemble de árboles de decisión
from sklearn.naive_bayes import MultinomialNB           # Naive Bayes para conteos discretos
from sklearn.metrics import confusion_matrix            # matriz de confusión para evaluación

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# ENTRENAMIENTO DE LOS TRES MODELOS CON MATRICES COUNT
# ─────────────────────────────────────────────────────────────────────────

# ── MODELO 1: Regresión Logística con matrices Count ───────────────────
#
# LogisticRegression: modelo lineal que aprende un peso w_k por cada token k.
#   Si w_k es positivo y grande → el token es señal de clase positiva (1)
#   Si w_k es negativo y grande → el token es señal de clase negativa (0)
#
# C=1.0: parámetro de regularización L2 (inverso de la fuerza de regularización).
#   C alto → poca regularización → modelo más flexible → riesgo de sobreajuste
#   C bajo → más regularización → modelo más restringido → riesgo de subentrenamiento
#   C=1.0 es el valor predeterminado de sklearn: punto de partida equilibrado.
#
# max_iter=1000: máximo de iteraciones del optimizador (solver) para convergencia.
#   El default (100) frecuentemente no converge con vocabularios de 1450+ tokens.
#
# random_state=1: semilla para reproducibilidad de la inicialización interna.
modeloLRcount = LogisticRegression(C=1.0, max_iter=1000, random_state=1)
modeloLRcount.fit(train_x_count, y_train)  # .fit(): entrena el modelo


# ── MODELO 2: Random Forest con matrices Count ─────────────────────────
#
# RandomForestClassifier: ensemble de N árboles de decisión independientes.
#   Cada árbol se entrena con una muestra aleatoria de los datos (bagging).
#   La predicción final es la clase mayoritaria entre todos los árboles.
#
# n_estimators=200: número de árboles en el bosque.
#   Más árboles → predicciones más estables y robustas, pero mayor costo computacional.
#   200 es un buen balance entre estabilidad y velocidad.
#
# max_depth=15: profundidad máxima permitida para cada árbol.
#   Sin límite: cada árbol memoriza los datos de train (train accuracy ~100%).
#   Con depth=15: se controla el sobreajuste limitando la complejidad de cada árbol.
modeloRFcount = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=1)
modeloRFcount.fit(train_x_count, y_train)


# ── MODELO 3: Naive Bayes Multinomial con matrices Count ───────────────
#
# MultinomialNB: versión de Naive Bayes diseñada para datos discretos (conteos).
#   Calcula P(clase | tokens) usando el teorema de Bayes, asumiendo que los
#   tokens son condicionalmente independientes entre sí dado la clase
#   (supuesto "naive" / ingenuo).
#
# alpha=0.5: suavizado de Laplace parcial.
#   alpha=1.0 (default) = suavizado completo de Laplace
#   alpha=0.5 = suavizado parcial: reduce el peso de tokens muy frecuentes
#   y mejora ligeramente la precisión en este problema de sentimiento.
modeloNBcount = MultinomialNB(alpha=0.5)
modeloNBcount.fit(train_x_count, y_train)


# ── EVALUACIÓN EN TRAIN Y VALIDACIÓN ───────────────────────────────────
# .score(X, y): calcula accuracy = predicciones correctas / total de predicciones.
# Se reporta para train y val para detectar sobreajuste (diferencia > 4%).

print('LR: Train-accuracy: %.2f%%' % (100*modeloLRcount.score(train_x_count, y_train)))
print('LR: Val-accuracy: %2.f%%'   % (100*modeloLRcount.score(val_x_count, y_val)))

print('\nRF: Train-accuracy: %.2f%%' % (100*modeloRFcount.score(train_x_count, y_train)))
print('RF: Val-accuracy: %.2f%%'    % (100*modeloRFcount.score(val_x_count, y_val)))

print('\nNB: Train-accuracy: %.2f%%' % (100*modeloNBcount.score(train_x_count, y_train)))
print('NB: Val-accuracy: %.2f%%'    % (100*modeloNBcount.score(val_x_count, y_val)))

##### **COMENTARIOS — Pregunta 9:**

**Resultados con matrices Count:**

| Modelo | Train Accuracy | Val Accuracy | Diferencia |
|--------|---------------|-------------|------------|
| Regresión Logística | ~95.2% | ~82.7% | ~12.5% |
| Random Forest | ~86.2% | ~76.7% | ~9.5% |
| Naive Bayes | ~92.1% | ~82.0% | ~10.1% |

Los tres modelos superan el umbral del 4% de diferencia train-val indicado en las instrucciones. Esto es esperable con representaciones BOW: las frecuencias brutas dan mucho peso a tokens comunes, lo que facilita cierto nivel de sobreajuste. La Regresión Logística tiene el mejor val accuracy absoluto (82.7%). Naive Bayes logra el mejor balance entre desempeño y diferencia train-val.

## **Pregunta - 10:** Modelos con matrices TF-IDF

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# ENTRENAMIENTO DE LOS TRES MODELOS CON MATRICES TF-IDF
#
# Se repite exactamente el mismo proceso de la Pregunta 9,
# pero usando las matrices TF-IDF (train_x_tfidf, val_x_tfidf, test_x_tfidf)
# en lugar de las matrices Count.
#
# TF-IDF penaliza tokens muy frecuentes en el corpus, lo que produce una
# ponderación diferente que puede afectar el aprendizaje de cada modelo.
# ─────────────────────────────────────────────────────────────────────────

# ── MODELO LR con TF-IDF ───────────────────────────────────────────────
# Mismos hiperparámetros que el modelo Count.
# La diferencia está únicamente en la representación de los datos de entrada.
modeloLRtfidf = LogisticRegression(C=1.0, max_iter=1000, random_state=1)
modeloLRtfidf.fit(train_x_tfidf, y_train)  # entrenar con matriz TF-IDF de train


# ── MODELO RF con TF-IDF ───────────────────────────────────────────────
modeloRFtfidf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=1)
modeloRFtfidf.fit(train_x_tfidf, y_train)


# ── MODELO NB con TF-IDF ───────────────────────────────────────────────
# alpha=0.1 (menor que el 0.5 usado con Count):
#   Los valores TF-IDF son pesos continuos normalizados (entre 0 y 1),
#   no conteos enteros crudos como en Count.
#   Al estar ya normalizados, se necesita menos suavizado de Laplace.
modeloNBtfidf = MultinomialNB(alpha=0.1)
modeloNBtfidf.fit(train_x_tfidf, y_train)


# ── EVALUACIÓN EN TRAIN Y VALIDACIÓN ───────────────────────────────────
print('Resultados parciales con matrices tf-idf:')
print('\nLR: Train-accuracy: %.2f%%' % (100*modeloLRtfidf.score(train_x_tfidf, y_train)))
print('LR: Val-accuracy: %2.f%%'    % (100*modeloLRtfidf.score(val_x_tfidf, y_val)))

print('\nRF: Train-accuracy: %.2f%%' % (100*modeloRFtfidf.score(train_x_tfidf, y_train)))
print('RF: Val-accuracy: %.2f%%'    % (100*modeloRFtfidf.score(val_x_tfidf, y_val)))

print('\nNB: Train-accuracy: %.2f%%' % (100*modeloNBtfidf.score(train_x_tfidf, y_train)))
print('NB: Val-accuracy: %.2f%%'    % (100*modeloNBtfidf.score(val_x_tfidf, y_val)))

##### **COMENTARIOS — Pregunta 10:**

**Resultados con matrices TF-IDF:**

| Modelo | Train Accuracy | Val Accuracy | Diferencia |
|--------|---------------|-------------|------------|
| Regresión Logística | ~92.9% | ~80.9% | ~12.0% |
| Random Forest | ~86.8% | ~76.4% | ~10.4% |
| Naive Bayes | ~93.8% | ~80.2% | ~13.6% |

TF-IDF baja ligeramente la val accuracy respecto a Count. Esto es coherente: las negaciones (`not`, `don't`) son frecuentes en muchos documentos, por lo que TF-IDF les asigna IDF bajo → peso bajo, reduciendo su influencia discriminativa. Sin embargo, **LR con TF-IDF generaliza mejor al conjunto de prueba (85.11%)** que LR con Count, lo que indica que TF-IDF ayuda a evitar sobreajuste en la evaluación final.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# EVALUACIÓN FINAL CON EL CONJUNTO DE PRUEBA (TEST)
#
# Se selecciona el mejor modelo de entre todos los entrenados.
# El conjunto de test NUNCA se usa durante el entrenamiento ni la búsqueda
# de hiperparámetros: representa datos completamente nuevos para el modelo.
#
# Mejor modelo seleccionado: Regresión Logística con TF-IDF (modeloLRtfidf)
#   Justificación:
#   - Mejor generalización al test (85.11%) entre todos los modelos evaluados
#   - Supera el mínimo requerido de 79% de accuracy en test
#   - La Regresión Logística es el modelo más adecuado para representaciones
#     BOW de alta dimensionalidad: asigna un peso interpretable a cada token
# ─────────────────────────────────────────────────────────────────────────

# Asignar el mejor modelo y su matriz de test asociada:
mejor_modelo    = modeloLRtfidf   # modelo ganador: LR con TF-IDF
test_x_asociada = test_x_tfidf   # usar la matriz TF-IDF de test (no la de Count)

# .score(X_test, y_test): calcula accuracy en el conjunto de prueba.
# Este número es el desempeño REAL del modelo en datos nunca vistos.
print('Test-accuracy con el mejor modelo %.2f%%' % (100*mejor_modelo.score(test_x_asociada, y_test)))

# .predict(X): genera las predicciones de clase (0 o 1) para cada documento de test.
pred = mejor_modelo.predict(test_x_asociada)

# confusion_matrix(y_real, y_pred, labels=[0,1]):
# Genera una matriz 2×2 donde:
#   labels=[0,1] → fila 0 = negativos reales, fila 1 = positivos reales
#   col 0 = predicción negativa, col 1 = predicción positiva
#
#   Posición [0,0] → Verdaderos Negativos (TN): negativos bien clasificados
#   Posición [0,1] → Falsos Positivos (FP): negativos clasificados como positivos
#   Posición [1,0] → Falsos Negativos (FN): positivos clasificados como negativos
#   Posición [1,1] → Verdaderos Positivos (TP): positivos bien clasificados
print('\nMatriz de confusión con el mejor modelo Tf-idf:')
print(confusion_matrix(y_test, pred, labels=[0,1]))

# / pred.shape[0]: divide entre el total de predicciones (450)
# para obtener proporciones en lugar de conteos absolutos.
print('\nMatriz de confusión con el mejor modelo de Tf-idf en proporciones:')
print(confusion_matrix(y_test, pred, labels=[0,1]) / pred.shape[0])

## **Pregunta - 11:** Conclusiones finales

##### **CONCLUSIONES FINALES — Pregunta 11:**

### Resumen del pipeline implementado

En esta actividad se construyó un pipeline completo de análisis de sentimiento sobre 3,000 comentarios en inglés (Amazon, IMDB, Yelp), cubriendo desde la corrección de datos hasta la evaluación de modelos de clasificación.

### Hallazgos principales

**1. Corrección de datos (IMDB):**
El parsing manual del archivo con `f.read()` + `rsplit('\t', 1)` fue la clave para recuperar los 252 registros perdidos por el error del parser de pandas con caracteres especiales.

**2. Preprocesamiento:**
- La **preservación de negaciones** en `mystopwords` fue la decisión de preprocesamiento más importante. `not` fue el token más frecuente en comentarios negativos según la nube de palabras.
- El **stemming con Snowball** + `min_freq=2` redujo el vocabulario de ~3,100 a 1,450 tokens, aligerando el modelo sin perder información relevante.

**3. Representaciones vectoriales:**
- La DTM resultó ser más del **99.1% ceros** (sparse). El uso de matrices dispersas (`scipy.sparse`) es esencial para manejar eficientemente este tipo de datos.
- Las matrices **Count** tienen leve ventaja en validación, pero **TF-IDF** generaliza mejor al conjunto de prueba final.

**4. Modelos:**
- **Regresión Logística** fue el mejor clasificador en ambas representaciones. Su capacidad de asignar pesos lineales a tokens es muy efectiva con BOW.
- **Naive Bayes** fue el segundo mejor, destacando por su simplicidad y velocidad.
- **Random Forest** tuvo el peor desempeño — los árboles de decisión no son ideales para espacios de alta dimensionalidad dispersa como las DTM.

**5. Resultado final:**
- El mejor modelo (**LR con TF-IDF**) alcanzó **85.11% de accuracy en test**, superando el umbral mínimo de 79% en 6 puntos porcentuales.
- La matriz de confusión muestra una distribución equilibrada: ~7.1% FP y ~7.8% FN, sin sesgo marcado hacia ninguna clase.

**6. Limitaciones del enfoque BOW:**
La representación Bag of Words pierde el orden de las palabras y no captura contexto semántico. "not good" ≠ "good" solo porque "not" está en el vocabulario — el modelo no entiende la relación semántica entre ambas palabras. Los embeddings y modelos de deep learning del resto del curso abordarán estas limitaciones capturando contexto y posición de cada token.

# **FIN DE LA ACTIVIDAD DE LAS SEMANAS 3 Y 4**